In [ ]:
# from google.colab import drive
# drive.mount("/content/drive") # Colab Specific

In [ ]:
!pip install torch==2.9.0+cu128 torchvision --extra-index-url https://download.pytorch.org/whl/cu128
!pip install transformers==5.1.0 datasets==4.5.0 scikit-learn==1.8.0 accelerate sentence-transformers evaluate

In [ ]:
import os, json, time, random
from pathlib import Path

In [ ]:
PROJECT_ROOT = "."
DATA_DIR     = f"{PROJECT_ROOT}/data"
OUT_ROOT     = f"{PROJECT_ROOT}/outputs/baseline"

In [ ]:
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("OUT_ROOT:", OUT_ROOT)

In [ ]:
import torch, transformers, datasets, sklearn
import numpy as np

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("sklearn:", sklearn.__version__)


In [ ]:
from datetime import datetime


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MODEL_NAME = "roberta-base"

MAX_LEN = 256
TRAIN_BS = 16
EVAL_BS = 32
LR = 2e-5
EPOCHS = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2
AGG_METHOD = "max"

RUN_NAME = f"baseline_{MODEL_NAME.replace('/','_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

RUN_DIR = f"{OUT_ROOT}/{RUN_NAME}"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

config = dict(
    run_name=RUN_NAME,
    model_name=MODEL_NAME,
    max_len=MAX_LEN,
    train_bs=TRAIN_BS,
    eval_bs=EVAL_BS,
    lr=LR,
    epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    seed=SEED,
    early_stop_patience=EARLY_STOP_PATIENCE,
    agg_method=AGG_METHOD,
    data_files=dict(
        train=f"{DATA_DIR}/train_cased.jsonl",
        validation=f"{DATA_DIR}/dev_cased.jsonl",
        test=f"{DATA_DIR}/test_cased.jsonl",
    )
    # data_files=dict(
    #     train=f"{DATA_DIR}/train.jsonl",
    #     validation=f"{DATA_DIR}/dev.jsonl",
    #     test=f"{DATA_DIR}/test.jsonl",
    # )
)

with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("RUN_DIR:", RUN_DIR)
print(json.dumps(config, indent=2)[:800], "...")


In [ ]:
from datasets import load_dataset

data_files = config["data_files"]
raw = load_dataset("json", data_files=data_files)

required = {"pid", "chunk_id", "text", "label"}
for split in raw.keys():
    cols = set(raw[split].column_names)
    missing = required - cols
    if missing:
        raise ValueError(f"Split '{split}' missing columns: {missing}")

def fix_types(ex):
    ex["pid"] = int(ex["pid"])
    ex["chunk_id"] = int(ex["chunk_id"])
    ex["label"] = int(ex["label"])
    return ex

raw = raw.map(fix_types)

def summarize(split):
    labels = np.array(raw[split]["label"])
    pids = np.array(raw[split]["pid"])
    uniq_pids = np.unique(pids)
    # patient-level label (assumes constant per pid)
    pid_to_label = {}
    for pid, lab in zip(pids, labels):
        pid_to_label.setdefault(pid, lab)
    pat_labels = np.array(list(pid_to_label.values()))
    return {
        "n_rows": len(labels),
        "n_patients": len(uniq_pids),
        "chunk_depressed_frac": float((labels==1).mean()),
        "patient_depressed_frac": float((pat_labels==1).mean()),
    }

stats = {s: summarize(s) for s in ["train", "validation", "test"]}
print(json.dumps(stats, indent=2))

with open(f"{RUN_DIR}/data_stats.json", "w") as f:
    json.dump(stats, f, indent=2)


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
collator = DataCollatorWithPadding(tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = raw.map(tok, batched=True)

tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label", "pid", "chunk_id"]
)

print(tokenized["train"][0])


In [ ]:
import evaluate
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

acc = evaluate.load("accuracy")

def chunk_metrics_from_probs(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    out = {
        "accuracy": float(acc.compute(predictions=preds, references=labels)["accuracy"]),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "auroc": float(roc_auc_score(labels, probs)) if len(np.unique(labels)) > 1 else float("nan"),
        "threshold": float(threshold),
    }
    return out

def aggregate_patient(probs, labels, pids, method="mean"):
    import pandas as pd
    df = pd.DataFrame({"pid": pids, "prob": probs, "label": labels})
    if method == "mean":
        agg = df.groupby("pid", as_index=False).agg(prob=("prob", "mean"), label=("label", "max"))
    elif method == "max":
        agg = df.groupby("pid", as_index=False).agg(prob=("prob", "max"), label=("label", "max"))
    else:
        raise ValueError("method must be 'mean' or 'max'")
    return agg

def patient_metrics_from_probs(probs, labels, pids, method="mean", threshold=0.5):
    agg = aggregate_patient(probs, labels, pids, method=method)
    return chunk_metrics_from_probs(agg["prob"].values, agg["label"].values, threshold=threshold), agg


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    return chunk_metrics_from_probs(probs, labels_np, threshold=0.5)

args = TrainingArguments(
    output_dir=f"{RUN_DIR}/checkpoints",
    eval_strategy="epoch",
    # save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="f1",
    greater_is_better=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=False,
    logging_steps=50,
    data_seed=SEED,
    seed=SEED,
    full_determinism=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)]
)


In [ ]:
train_out = trainer.train()
# trainer.save_model(f"{RUN_DIR}/best_model")
# print("Saved best model to:", f"{RUN_DIR}/best_model")


In [ ]:
dev_chunk = trainer.evaluate(tokenized["validation"])
test_chunk = trainer.evaluate(tokenized["test"])

print("DEV chunk metrics:", dev_chunk)
print("TEST chunk metrics:", test_chunk)

with open(f"{RUN_DIR}/chunk_metrics.json", "w") as f:
    json.dump({"dev": dev_chunk, "test": test_chunk}, f, indent=2)


In [ ]:
import pandas as pd

def predict_split(split_name):
    pred = trainer.predict(tokenized[split_name])
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    labels_np = np.array(raw[split_name]["label"])
    pids = np.array(raw[split_name]["pid"])
    chunk_ids = np.array(raw[split_name]["chunk_id"])

    df = pd.DataFrame({
        "pid": pids,
        "chunk_id": chunk_ids,
        "prob_depressed": probs,
        "label": labels_np
    })
    return df

dev_df = predict_split("validation")
test_df = predict_split("test")

dev_df.to_csv(f"{RUN_DIR}/dev_chunk_preds.csv", index=False)
test_df.to_csv(f"{RUN_DIR}/test_chunk_preds.csv", index=False)

print(dev_df.head())


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

_, dev_pat_df = patient_metrics_from_probs(
    probs=dev_df["prob_depressed"].values,
    labels=dev_df["label"].values,
    pids=dev_df["pid"].values,
    method=AGG_METHOD,
    threshold=0.5
)

dev_probs = dev_pat_df["prob"].values
dev_labels = dev_pat_df["label"].values

best_f1 = -1
best_threshold = 0.5

for t in np.linspace(0.05, 0.95, 181):  # step 0.005
    preds = (dev_probs >= t).astype(int)
    f1 = f1_score(dev_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(t)

print("Best threshold from DEV:", best_threshold)
print("Best DEV F1:", best_f1)

dev_pat_metrics, dev_pat_df = patient_metrics_from_probs(
    probs=dev_df["prob_depressed"].values,
    labels=dev_df["label"].values,
    pids=dev_df["pid"].values,
    method=AGG_METHOD,
    threshold=best_threshold
)

test_pat_metrics, test_pat_df = patient_metrics_from_probs(
    probs=test_df["prob_depressed"].values,
    labels=test_df["label"].values,
    pids=test_df["pid"].values,
    method=AGG_METHOD,
    threshold=best_threshold
)

print("DEV patient metrics (tuned):", dev_pat_metrics)
print("TEST patient metrics (tuned):", test_pat_metrics)

dev_pat_df.to_csv(f"{RUN_DIR}/dev_patient_preds.csv", index=False)
test_pat_df.to_csv(f"{RUN_DIR}/test_patient_preds.csv", index=False)

with open(f"{RUN_DIR}/patient_metrics.json", "w") as f:
    json.dump(
        {"threshold": best_threshold, "dev": dev_pat_metrics, "test": test_pat_metrics},
        f,
        indent=2
    )


In [ ]:
summary = {
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "data_stats": stats,
    "chunk_metrics": {"dev": dev_chunk, "test": test_chunk},
    "patient_metrics": {"dev": dev_pat_metrics, "test": test_pat_metrics},
    "agg_method": AGG_METHOD,
    "threshold": best_threshold,
    "dev_best_f1": best_f1,
    "notes": "Baseline (no retrieval). Patient-level is main. Chunking: 150 tokens, stride 75. Threshold tuned on dev."
}

with open(f"{RUN_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote:", f"{RUN_DIR}/summary.json")
print(json.dumps(summary, indent=2)[:1200], "...")
